In [ ]:
import json
import random
import pandas as pd
import os
import argparse

def generate_labeled_requests(policy_json, num_allow=3, num_deny=2):
    policy = json.loads(policy_json)
    statements = policy.get("Statement", [])
    requests = []

    # Helper functions to extract and normalize values
    def get_action(stmt):
        action = stmt.get("Action")
        if action is None:
            # If no action specified, extract service from other actions in policy or use generic
            all_actions = []
            for s in statements:
                stmt_actions = s.get("Action", [])
                if isinstance(stmt_actions, list):
                    all_actions.extend(stmt_actions)
                elif stmt_actions:
                    all_actions.append(stmt_actions)
            
            if all_actions:
                # Use an action from the policy
                return random.choice(all_actions)
            else:
                # Fallback to generic action
                return "*"
        
        # Preserve list structure
        if isinstance(action, list):
            return action  # Return the entire list
        return action  # Return single string

    def get_resource(stmt):
        resource = stmt.get("Resource")
        if resource is None:
            # If no resource specified, extract from other statements or use generic
            all_resources = []
            for s in statements:
                stmt_resources = s.get("Resource", [])
                if isinstance(stmt_resources, list):
                    all_resources.extend(stmt_resources)
                elif stmt_resources:
                    all_resources.append(stmt_resources)
            
            if all_resources:
                # Use a resource from the policy
                resource = random.choice(all_resources)
            else:
                # Fallback to generic resource
                resource = "*"
        
        # Preserve list structure
        if isinstance(resource, list):
            # Process each resource in the list
            processed_resources = []
            for res in resource:
                if "*" in res:
                    processed_resources.append(res.replace("*", f"object-{random.randint(1, 100)}"))
                else:
                    processed_resources.append(res)
            return processed_resources
        else:
            # Single resource
            if "*" in resource:
                return resource.replace("*", f"object-{random.randint(1, 100)}")
            return resource

    def get_principal(stmt):
        principal = stmt.get("Principal")
        if principal is None:
            # If no principal specified, use a generic one
            return "arn:aws:iam::123456789012:user/generic-user"
        
        if isinstance(principal, dict):
            # Handle {"AWS": "arn:..."} or {"AWS": ["arn1", "arn2"]} format
            values = list(principal.values())[0]
            return values  # Preserve list structure if it exists
        return principal  # Return as-is (could be string or list)

    def get_condition(stmt):
        cond_block = stmt.get("Condition", {})
        if not cond_block:
            # If no conditions in this statement, return empty dict (no default condition)
            return {}
        
        flattened = {}
        for operator, conds in cond_block.items():
            for key, val in conds.items():
                if isinstance(val, (int, float)):
                    flattened[key] = val + 1  # Slightly modify to ensure it should still pass
                else:
                    flattened[key] = val
        return flattened

    # Generate allowed requests
    for _ in range(num_allow):
        stmt = random.choice([s for s in statements if s.get("Effect", "").lower() == "allow"])
        request = {
            "Effect": "allow",
            "Principal": get_principal(stmt),
            "Action": get_action(stmt),
            "Resource": get_resource(stmt),
            "ExpectedDecision": "Allow"
        }
        
        # Only add condition if it exists
        condition = get_condition(stmt)
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    # Generate denied requests
    for _ in range(num_deny):
        stmt = random.choice([s for s in statements if s.get("Effect", "").lower() == "allow"])
        action = get_action(stmt)
        resource = get_resource(stmt)
        principal = get_principal(stmt)
        condition = get_condition(stmt)
        
        # Choose what to violate
        violation_type = random.choice(["action", "resource", "principal", "condition"])
        
        if violation_type == "action":
            # Modify action(s)
            if isinstance(action, list):
                # Add invalid actions to the list or modify existing ones
                modified_actions = []
                for act in action:
                    if random.random() < 0.7:  # 70% chance to make invalid
                        modified_actions.append(act + ".Invalid")
                    else:
                        modified_actions.append(act)
                # Also add some completely forbidden actions
                service = action[0].split(':')[0] if action else "s3"
                forbidden_actions = {
                    "s3": ["s3:DeleteBucket", "s3:DeleteObject"],
                    "logs": ["logs:DeleteLogGroup", "logs:PutRetentionPolicy"],
                    "ec2": ["ec2:TerminateInstances", "ec2:DeleteSecurityGroup"],
                    "iam": ["iam:DeleteUser", "iam:CreateRole"]
                }
                if service in forbidden_actions:
                    modified_actions.extend(random.sample(forbidden_actions[service], 1))
                action = modified_actions
            else:
                # Single action - make it invalid
                action = action + ".Invalid"
                
        elif violation_type == "resource":
            # Modify resource(s)
            if isinstance(resource, list):
                modified_resources = []
                for res in resource:
                    # Change to unauthorized resource
                    if "s3:::" in res:
                        modified_resources.append("arn:aws:s3:::unauthorized-bucket/forbidden-object")
                    elif "logs:" in res:
                        modified_resources.append("arn:aws:logs:us-east-1:999999999:log-group:forbidden-logs:*")
                    else:
                        modified_resources.append("arn:aws:s3:::invalid-bucket/unknown-object")
                resource = modified_resources
            else:
                # Single resource
                if "s3:::" in resource:
                    resource = "arn:aws:s3:::unauthorized-bucket/forbidden-object"
                elif "logs:" in resource:
                    resource = "arn:aws:logs:us-east-1:999999999:log-group:forbidden-logs:*"
                else:
                    resource = "arn:aws:s3:::invalid-bucket/unknown-object"
                    
        elif violation_type == "principal":
            # Modify principal
            if isinstance(principal, list):
                principal = ["arn:aws:iam::999999999:user/unauthorized-user"]
            else:
                principal = "arn:aws:iam::999999999:user/unauthorized-user"
                
        elif violation_type == "condition":
            # Violate condition constraints
            violated_condition = {}
            for key, val in condition.items():
                if isinstance(val, (int, float)):
                    # Violate numeric conditions
                    if "MaxKeys" in key:
                        violated_condition[key] = val + 50  # Exceed the limit
                    else:
                        violated_condition[key] = val * 10  # Make it much larger
                else:
                    violated_condition[key] = "InvalidValue"
            condition = violated_condition
        
        request = {
            "Effect": "allow",  # Still requesting allow, but should be denied
            "Principal": principal,
            "Action": action,
            "Resource": resource,
            "ExpectedDecision": "Deny"
        }
        
        # Only add condition if it exists
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    return requests

def save_request_to_file(request, filepath):
    """Save a single request to a JSON file"""
    # Remove ExpectedDecision from output
    clean_req = {k: v for k, v in request.items() if k != 'ExpectedDecision'}
    
    # Only include fields that have values
    final_req = {}
    for key in ["Effect", "Principal", "Action", "Resource", "Condition"]:
        if key in clean_req and clean_req[key]:
            final_req[key] = clean_req[key]
    
    # Format as single request JSON
    single_request = {"Requests": [final_req]}
    
    with open(filepath, 'w') as f:
        json.dump(single_request, f, indent=4)

def process_policy(policy_json, policy_index, output_dir):
    """Process a single policy and save requests to files"""
    print(f"Processing policy {policy_index}...")
    
    # Generate requests (3 allowed, 2 denied)
    requests = generate_labeled_requests(policy_json, num_allow=3, num_deny=2)
    
    # Separate allowed and denied requests
    allowed_requests = [req for req in requests if req['ExpectedDecision'] == 'Allow']
    denied_requests = [req for req in requests if req['ExpectedDecision'] == 'Deny']
    
    print(f"  Generated {len(allowed_requests)} allowed and {len(denied_requests)} denied requests")

    # Save allowed requests to files 90.json, 91.json, 92.json
    allowed_file_numbers = [90, 91, 92]
    for i, request in enumerate(allowed_requests):
        if i < len(allowed_file_numbers):
            file_number = allowed_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved allowed request to {filepath}")
    
    # Save denied requests to files 63.json, 64.json
    denied_file_numbers = [93, 94]
    for i, request in enumerate(denied_requests):
        if i < len(denied_file_numbers):
            file_number = denied_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved denied request to {filepath}")

def main():
    parser = argparse.ArgumentParser(description='Generate requests for multiple policies')
    parser.add_argument('--policies', '-p', nargs='+', required=True, 
                       help='Policy JSON strings or file paths')
    parser.add_argument('--output-dir', '-o', 
                       default='/home/bhall2/Documents/fixmypolicy/FL/Dataset/requests',
                       help='Output directory for request files')
    
    args = parser.parse_args()
    
    # Create output directory if it doesn't exist
    os.makedirs(args.output_dir, exist_ok=True)
    
    for i, policy_input in enumerate(args.policies):
        # Check if it's a file path or JSON string
        if os.path.isfile(policy_input):
            with open(policy_input, 'r') as f:
                policy_json = f.read()
        else:
            policy_json = policy_input
        
        # Create subdirectory for this policy
        policy_dir = os.path.join(args.output_dir, f"policy_{i}")
        os.makedirs(policy_dir, exist_ok=True)
        
        try:
            process_policy(policy_json, i, policy_dir)
        except Exception as e:
            print(f"Error processing policy {i}: {e}")
            continue
    
    print("All policies processed!")
    
    import json
import random
import pandas as pd
import os
import argparse

def generate_labeled_requests(policy_json, num_allow=3, num_deny=2):
    policy = json.loads(policy_json)
    statements = policy.get("Statement", [])
    requests = []

    # Helper functions to extract and normalize values
    def get_action(stmt):
        action = stmt.get("Action")
        if action is None:
            # If no action specified, extract service from other actions in policy or use generic
            all_actions = []
            for s in statements:
                stmt_actions = s.get("Action", [])
                if isinstance(stmt_actions, list):
                    all_actions.extend(stmt_actions)
                elif stmt_actions:
                    all_actions.append(stmt_actions)
            
            if all_actions:
                # Use an action from the policy
                return random.choice(all_actions)
            else:
                # Fallback to generic action
                return "*"
        
        # Preserve list structure
        if isinstance(action, list):
            return action  # Return the entire list
        return action  # Return single string

    def get_resource(stmt):
        resource = stmt.get("Resource")
        if resource is None:
            # If no resource specified, extract from other statements or use generic
            all_resources = []
            for s in statements:
                stmt_resources = s.get("Resource", [])
                if isinstance(stmt_resources, list):
                    all_resources.extend(stmt_resources)
                elif stmt_resources:
                    all_resources.append(stmt_resources)
            
            if all_resources:
                # Use a resource from the policy
                resource = random.choice(all_resources)
            else:
                # Fallback to generic resource
                resource = "*"
        
        # Preserve list structure
        if isinstance(resource, list):
            # Process each resource in the list
            processed_resources = []
            for res in resource:
                if "*" in res:
                    processed_resources.append(res.replace("*", f"object-{random.randint(1, 100)}"))
                else:
                    processed_resources.append(res)
            return processed_resources
        else:
            # Single resource
            if "*" in resource:
                return resource.replace("*", f"object-{random.randint(1, 100)}")
            return resource

    def get_principal(stmt):
        principal = stmt.get("Principal")
        if principal is None:
            # If no principal specified, use a generic one
            return "arn:aws:iam::123456789012:user/generic-user"
        
        if isinstance(principal, dict):
            # Handle {"AWS": "arn:..."} or {"AWS": ["arn1", "arn2"]} format
            values = list(principal.values())[0]
            return values  # Preserve list structure if it exists
        return principal  # Return as-is (could be string or list)

    def get_condition(stmt):
        cond_block = stmt.get("Condition", {})
        if not cond_block:
            # If no conditions in this statement, return empty dict (no default condition)
            return {}
        
        flattened = {}
        for operator, conds in cond_block.items():
            for key, val in conds.items():
                if isinstance(val, (int, float)):
                    flattened[key] = val + 1  # Slightly modify to ensure it should still pass
                else:
                    flattened[key] = val
        return flattened

    # Generate allowed requests
    for _ in range(num_allow):
        stmt = random.choice([s for s in statements if s.get("Effect", "").lower() == "allow"])
        request = {
            "Effect": "allow",
            "Principal": get_principal(stmt),
            "Action": get_action(stmt),
            "Resource": get_resource(stmt),
            "ExpectedDecision": "Allow"
        }
        
        # Only add condition if it exists
        condition = get_condition(stmt)
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    # Generate denied requests
    for _ in range(num_deny):
        stmt = random.choice([s for s in statements if s.get("Effect", "").lower() == "allow"])
        action = get_action(stmt)
        resource = get_resource(stmt)
        principal = get_principal(stmt)
        condition = get_condition(stmt)
        
        # Choose what to violate
        violation_type = random.choice(["action", "resource", "principal", "condition"])
        
        if violation_type == "action":
            # Modify action(s)
            if isinstance(action, list):
                # Add invalid actions to the list or modify existing ones
                modified_actions = []
                for act in action:
                    if random.random() < 0.7:  # 70% chance to make invalid
                        modified_actions.append(act + ".Invalid")
                    else:
                        modified_actions.append(act)
                # Also add some completely forbidden actions
                service = action[0].split(':')[0] if action else "s3"
                forbidden_actions = {
                    "s3": ["s3:DeleteBucket", "s3:DeleteObject"],
                    "logs": ["logs:DeleteLogGroup", "logs:PutRetentionPolicy"],
                    "ec2": ["ec2:TerminateInstances", "ec2:DeleteSecurityGroup"],
                    "iam": ["iam:DeleteUser", "iam:CreateRole"]
                }
                if service in forbidden_actions:
                    modified_actions.extend(random.sample(forbidden_actions[service], 1))
                action = modified_actions
            else:
                # Single action - make it invalid
                action = action + ".Invalid"
                
        elif violation_type == "resource":
            # Modify resource(s)
            if isinstance(resource, list):
                modified_resources = []
                for res in resource:
                    # Change to unauthorized resource
                    if "s3:::" in res:
                        modified_resources.append("arn:aws:s3:::unauthorized-bucket/forbidden-object")
                    elif "logs:" in res:
                        modified_resources.append("arn:aws:logs:us-east-1:999999999:log-group:forbidden-logs:*")
                    else:
                        modified_resources.append("arn:aws:s3:::invalid-bucket/unknown-object")
                resource = modified_resources
            else:
                # Single resource
                if "s3:::" in resource:
                    resource = "arn:aws:s3:::unauthorized-bucket/forbidden-object"
                elif "logs:" in resource:
                    resource = "arn:aws:logs:us-east-1:999999999:log-group:forbidden-logs:*"
                else:
                    resource = "arn:aws:s3:::invalid-bucket/unknown-object"
                    
        elif violation_type == "principal":
            # Modify principal
            if isinstance(principal, list):
                principal = ["arn:aws:iam::999999999:user/unauthorized-user"]
            else:
                principal = "arn:aws:iam::999999999:user/unauthorized-user"
                
        elif violation_type == "condition":
            # Violate condition constraints
            violated_condition = {}
            for key, val in condition.items():
                if isinstance(val, (int, float)):
                    # Violate numeric conditions
                    if "MaxKeys" in key:
                        violated_condition[key] = val + 50  # Exceed the limit
                    else:
                        violated_condition[key] = val * 10  # Make it much larger
                else:
                    violated_condition[key] = "InvalidValue"
            condition = violated_condition
        
        request = {
            "Effect": "allow",  # Still requesting allow, but should be denied
            "Principal": principal,
            "Action": action,
            "Resource": resource,
            "ExpectedDecision": "Deny"
        }
        
        # Only add condition if it exists
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    return requests

def save_request_to_file(request, filepath):
    """Save a single request to a JSON file"""
    # Remove ExpectedDecision from output
    clean_req = {k: v for k, v in request.items() if k != 'ExpectedDecision'}
    
    # Only include fields that have values
    final_req = {}
    for key in ["Effect", "Principal", "Action", "Resource", "Condition"]:
        if key in clean_req and clean_req[key]:
            final_req[key] = clean_req[key]
    
    # Format as single request JSON
    single_request = {"Requests": [final_req]}
    
    with open(filepath, 'w') as f:
        json.dump(single_request, f, indent=4)

def process_policy(policy_json, policy_index, output_dir):
    """Process a single policy and save requests to files"""
    print(f"Processing policy {policy_index}...")
    
    # Generate requests (3 allowed, 2 denied)
    requests = generate_labeled_requests(policy_json, num_allow=3, num_deny=2)
    
    # Separate allowed and denied requests
    allowed_requests = [req for req in requests if req['ExpectedDecision'] == 'Allow']
    denied_requests = [req for req in requests if req['ExpectedDecision'] == 'Deny']
    
    print(f"  Generated {len(allowed_requests)} allowed and {len(denied_requests)} denied requests")

    # Save allowed requests to files 00.json, 01.json, 02.json
    allowed_file_numbers = ["00", "01", "02"]
    for i, request in enumerate(allowed_requests):
        if i < len(allowed_file_numbers):
            file_number = allowed_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved allowed request to {filepath}")
    
    # Save denied requests to files 63.json, 64.json
    denied_file_numbers = [93, 94]
    for i, request in enumerate(denied_requests):
        if i < len(denied_file_numbers):
            file_number = denied_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved denied request to {filepath}")

def main():
    parser = argparse.ArgumentParser(description='Generate requests for multiple policies')
    parser.add_argument('--policies', '-p', nargs='+', required=True, 
                       help='Policy JSON strings or file paths')
    parser.add_argument('--output-dir', '-o', 
                       default='/home/bhall2/Documents/fixmypolicy/FL/Dataset/requests',
                       help='Output directory for request files')
    
    args = parser.parse_args()
    
    # Create output directory if it doesn't exist
    os.makedirs(args.output_dir, exist_ok=True)
    
    for i, policy_input in enumerate(args.policies):
        # Check if it's a file path or JSON string
        if os.path.isfile(policy_input):
            with open(policy_input, 'r') as f:
                policy_json = f.read()
        else:
            policy_json = policy_input
        
        # Create subdirectory for this policy
        policy_dir = os.path.join(args.output_dir, f"policy_{i}")
        os.makedirs(policy_dir, exist_ok=True)
        
        try:
            process_policy(policy_json, i, policy_dir)
        except Exception as e:
            print(f"Error processing policy {i}: {e}")
            continue
    
    print("All policies processed!")

    
example_policy = '''
{
  "Statement": [
    {
      "Action": [
        "s3:Get*",
        "s3:List*",
        "s3:PutObject",
        "s3:DeleteObject"
      ],
      "Resource": "arn:aws:s3:::athena-query-results/*",
      "Effect": "Allow",
      "Sid": "AllowS3AccessToSaveAndReadQueryResults"
    },
    {
      "Action": [
        "s3:*"
      ],
      "Resource": "arn:aws:s3:::bkt_logs/*",
      "Effect": "Allow",
      "Sid": "AllowS3AccessForGlueToReadLogs"
    },
    {
      "Action": [
        "athena:GetQueryExecution",
        "athena:StartQueryExecution",
        "athena:StopQueryExecution",
        "athena:GetWorkGroup",
        "athena:GetDatabase",
        "athena:BatchGetQueryExecution",
        "athena:GetQueryResults",
        "athena:GetQueryResultsStream",
        "athena:GetTableMetadata"
      ], 
      "Resource": [
        "*"
      ],
      "Effect": "Allow",
      "Sid": "AllowAthenaAccess"
    },
    {
      "Action": [
        "glue:GetTable",
        "glue:GetDatabase",
        "glue:GetPartitions"
      ],
      "Resource": [
        "*"
      ],
      "Effect": "Allow",
      "Sid": "AllowGlueAccess"
    },
    {
      "Action": [
        "kms:CreateGrant",
        "kms:DescribeKey",
        "kms:Decrypt"
      ],
      "Resource": [
        "*"
      ],
      "Effect": "Allow",
      "Sid": "AllowKMSAccess"
    }
  ]
}
'''
 
output_dir = '/home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2'
os.makedirs(output_dir, exist_ok=True)
process_policy(example_policy, 0, output_dir)

Processing policy 0...
  Generated 3 allowed and 2 denied requests
  Saved allowed request to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/00.json
  Saved allowed request to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/01.json
  Saved allowed request to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/02.json
  Saved denied request to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/93.json
  Saved denied request to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/94.json


In [6]:
import json
import random
import pandas as pd
import os
import argparse

def generate_labeled_requests(policy_json, num_allow=3, num_deny=2):
    policy = json.loads(policy_json)
    statements = policy.get("Statement", [])
    requests = []

    # Helper functions to extract and normalize values
    def get_action(stmt):
        action = stmt.get("Action")
        if action is None:
            # If no action specified, extract service from other actions in policy or use generic
            all_actions = []
            for s in statements:
                stmt_actions = s.get("Action", [])
                if isinstance(stmt_actions, list):
                    all_actions.extend(stmt_actions)
                elif stmt_actions:
                    all_actions.append(stmt_actions)
            
            if all_actions:
                # Use an action from the policy
                return random.choice(all_actions)
            else:
                # Fallback to generic action
                return "*"
        
        # Preserve list structure
        if isinstance(action, list):
            return action  # Return the entire list
        return action  # Return single string

    def get_resource(stmt):
        resource = stmt.get("Resource")
        if resource is None:
            # If no resource specified, extract from other statements or use generic
            all_resources = []
            for s in statements:
                stmt_resources = s.get("Resource", [])
                if isinstance(stmt_resources, list):
                    all_resources.extend(stmt_resources)
                elif stmt_resources:
                    all_resources.append(stmt_resources)
            
            if all_resources:
                # Use a resource from the policy
                resource = random.choice(all_resources)
            else:
                # Fallback to generic resource
                resource = "*"
        
        # Preserve list structure
        if isinstance(resource, list):
            # Process each resource in the list
            processed_resources = []
            for res in resource:
                if "*" in res:
                    processed_resources.append(res.replace("*", f"object-{random.randint(1, 100)}"))
                else:
                    processed_resources.append(res)
            return processed_resources
        else:
            # Single resource
            if "*" in resource:
                return resource.replace("*", f"object-{random.randint(1, 100)}")
            return resource

    def get_principal(stmt):
        principal = stmt.get("Principal")
        if principal is None:
            # If no principal specified, use a generic one
            return "arn:aws:iam::123456789012:user/generic-user"
        
        if isinstance(principal, dict):
            # Handle {"AWS": "arn:..."} or {"AWS": ["arn1", "arn2"]} format
            values = list(principal.values())[0]
            return values  # Preserve list structure if it exists
        return principal  # Return as-is (could be string or list)

    def get_condition(stmt):
        cond_block = stmt.get("Condition", {})
        if not cond_block:
            # If no conditions in this statement, return empty dict (no default condition)
            return {}
        
        flattened = {}
        for operator, conds in cond_block.items():
            for key, val in conds.items():
                if isinstance(val, (int, float)):
                    flattened[key] = val + 1  # Slightly modify to ensure it should still pass
                else:
                    flattened[key] = val
        return flattened

    # Get all Allow statements
    allow_statements = [s for s in statements if s.get("Effect", "").lower() == "allow"]
    
    if not allow_statements:
        print("Warning: No Allow statements found in policy")
        return []

    # Generate allowed requests - ensure we cover different statements
    used_statements = []
    for i in range(num_allow):
        # Try to use different statements, but allow repeating if we run out
        if i < len(allow_statements):
            stmt = allow_statements[i]
        else:
            stmt = random.choice(allow_statements)
        
        used_statements.append(stmt)
        
        request = {
            "Effect": "allow",
            "Principal": get_principal(stmt),
            "Action": get_action(stmt),
            "Resource": get_resource(stmt),
            "ExpectedDecision": "Allow"
        }
        
        # Only add condition if it exists
        condition = get_condition(stmt)
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    # Generate denied requests - also try to cover different statements
    for i in range(num_deny):
        # Try to use different statements for variety
        if i < len(allow_statements):
            stmt = allow_statements[i]
        else:
            stmt = random.choice(allow_statements)
            
        action = get_action(stmt)
        resource = get_resource(stmt)
        principal = get_principal(stmt)
        condition = get_condition(stmt)
        
        # Choose what to violate
        violation_type = random.choice(["action", "resource", "principal", "condition"])
        
        if violation_type == "action":
            # Modify action(s)
            if isinstance(action, list):
                # Add invalid actions to the list or modify existing ones
                modified_actions = []
                for act in action:
                    if random.random() < 0.7:  # 70% chance to make invalid
                        modified_actions.append(act + ".Invalid")
                    else:
                        modified_actions.append(act)
                # Also add some completely forbidden actions
                service = action[0].split(':')[0] if action else "s3"
                forbidden_actions = {
                    "s3": ["s3:DeleteBucket", "s3:DeleteObject"],
                    "logs": ["logs:DeleteLogGroup", "logs:PutRetentionPolicy"],
                    "ec2": ["ec2:TerminateInstances", "ec2:DeleteSecurityGroup"],
                    "iam": ["iam:DeleteUser", "iam:CreateRole"]
                }
                if service in forbidden_actions:
                    modified_actions.extend(random.sample(forbidden_actions[service], 1))
                action = modified_actions
            else:
                # Single action - make it invalid
                action = action + ".Invalid"
                
        elif violation_type == "resource":
            # Modify resource(s)
            if isinstance(resource, list):
                modified_resources = []
                for res in resource:
                    # Change to unauthorized resource
                    if "s3:::" in res:
                        modified_resources.append("arn:aws:s3:::unauthorized-bucket/forbidden-object")
                    elif "logs:" in res:
                        modified_resources.append("arn:aws:logs:us-east-1:999999999:log-group:forbidden-logs:*")
                    else:
                        modified_resources.append("arn:aws:s3:::invalid-bucket/unknown-object")
                resource = modified_resources
            else:
                # Single resource
                if "s3:::" in resource:
                    resource = "arn:aws:s3:::unauthorized-bucket/forbidden-object"
                elif "logs:" in resource:
                    resource = "arn:aws:logs:us-east-1:999999999:log-group:forbidden-logs:*"
                else:
                    resource = "arn:aws:s3:::invalid-bucket/unknown-object"
                    
        elif violation_type == "principal":
            # Modify principal
            if isinstance(principal, list):
                principal = ["arn:aws:iam::999999999:user/unauthorized-user"]
            else:
                principal = "arn:aws:iam::999999999:user/unauthorized-user"
                
        elif violation_type == "condition":
            # Violate condition constraints
            violated_condition = {}
            for key, val in condition.items():
                if isinstance(val, (int, float)):
                    # Violate numeric conditions
                    if "MaxKeys" in key:
                        violated_condition[key] = val + 50  # Exceed the limit
                    else:
                        violated_condition[key] = val * 10  # Make it much larger
                else:
                    violated_condition[key] = "InvalidValue"
            condition = violated_condition
        
        request = {
            "Effect": "allow",  # Still requesting allow, but should be denied
            "Principal": principal,
            "Action": action,
            "Resource": resource,
            "ExpectedDecision": "Deny"
        }
        
        # Only add condition if it exists
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    return requests

def generate_comprehensive_requests(policy_json):
    """Generate requests that systematically cover each statement in the policy"""
    policy = json.loads(policy_json)
    statements = policy.get("Statement", [])
    requests = []

    # Helper functions (same as above)
    def get_action(stmt):
        action = stmt.get("Action")
        if action is None:
            return "*"
        return action

    def get_resource(stmt):
        resource = stmt.get("Resource")
        if resource is None:
            return "*"
        
        if isinstance(resource, list):
            processed_resources = []
            for res in resource:
                if "*" in res:
                    processed_resources.append(res.replace("*", f"object-{random.randint(1, 100)}"))
                else:
                    processed_resources.append(res)
            return processed_resources
        else:
            if "*" in resource:
                return resource.replace("*", f"object-{random.randint(1, 100)}")
            return resource

    def get_principal(stmt):
        principal = stmt.get("Principal")
        if principal is None:
            return "arn:aws:iam::123456789012:user/generic-user"
        
        if isinstance(principal, dict):
            values = list(principal.values())[0]
            return values
        return principal

    def get_condition(stmt):
        cond_block = stmt.get("Condition", {})
        if not cond_block:
            return {}
        
        flattened = {}
        for operator, conds in cond_block.items():
            for key, val in conds.items():
                if isinstance(val, (int, float)):
                    flattened[key] = val + 1
                else:
                    flattened[key] = val
        return flattened

    # Generate one allowed request per statement
    allow_statements = [s for s in statements if s.get("Effect", "").lower() == "allow"]
    
    for i, stmt in enumerate(allow_statements):
        request = {
            "Effect": "allow",
            "Principal": get_principal(stmt),
            "Action": get_action(stmt),
            "Resource": get_resource(stmt),
            "ExpectedDecision": "Allow",
            "StatementIndex": i,  # Track which statement this comes from
            "StatementSid": stmt.get("Sid", f"Statement{i}")
        }
        
        condition = get_condition(stmt)
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    # Generate denied requests by violating each statement
    for i, stmt in enumerate(allow_statements[:2]):  # Limit to first 2 for denied requests
        action = get_action(stmt)
        resource = get_resource(stmt)
        principal = get_principal(stmt)
        condition = get_condition(stmt)
        
        # Create a violation based on the statement type
        if isinstance(action, list) and any("s3:" in act for act in action):
            # For S3 actions, violate by using unauthorized bucket
            if isinstance(resource, list):
                resource = ["arn:aws:s3:::unauthorized-bucket/forbidden-object"]
            else:
                resource = "arn:aws:s3:::unauthorized-bucket/forbidden-object"
        elif isinstance(action, list) and any("athena:" in act for act in action):
            # For Athena actions, add forbidden action
            action = action + ["athena:DeleteWorkGroup"]
        else:
            # Generic violation - add invalid action
            if isinstance(action, list):
                action = action + [action[0] + ".Invalid"]
            else:
                action = action + ".Invalid"
        
        request = {
            "Effect": "allow",
            "Principal": principal,
            "Action": action,
            "Resource": resource,
            "ExpectedDecision": "Deny",
            "StatementIndex": i,
            "StatementSid": stmt.get("Sid", f"Statement{i}"),
            "ViolationType": "resource" if "unauthorized" in str(resource) else "action"
        }
        
        if condition:
            request["Condition"] = condition
            
        requests.append(request)

    return requests

def save_request_to_file(request, filepath):
    """Save a single request to a JSON file"""
    # Remove ExpectedDecision, StatementIndex, StatementSid, ViolationType from output
    clean_req = {k: v for k, v in request.items() 
                 if k not in ['ExpectedDecision', 'StatementIndex', 'StatementSid', 'ViolationType']}
    
    # Only include fields that have values
    final_req = {}
    for key in ["Effect", "Principal", "Action", "Resource", "Condition"]:
        if key in clean_req and clean_req[key]:
            final_req[key] = clean_req[key]
    
    # Format as single request JSON
    single_request = {"Requests": [final_req]}
    
    with open(filepath, 'w') as f:
        json.dump(single_request, f, indent=4)

def process_policy_comprehensive(policy_json, policy_index, output_dir):
    """Process a single policy and generate comprehensive requests covering all statements"""
    print(f"Processing policy {policy_index} comprehensively...")
    
    # Generate comprehensive requests covering all statements
    requests = generate_comprehensive_requests(policy_json)
    
    # Separate allowed and denied requests
    allowed_requests = [req for req in requests if req['ExpectedDecision'] == 'Allow']
    denied_requests = [req for req in requests if req['ExpectedDecision'] == 'Deny']
    
    print(f"  Generated {len(allowed_requests)} allowed and {len(denied_requests)} denied requests")
    print(f"  Allowed requests cover statements: {[req.get('StatementSid', 'Unknown') for req in allowed_requests]}")
    print(f"  Denied requests cover statements: {[req.get('StatementSid', 'Unknown') for req in denied_requests]}")

    # Save allowed requests
    allowed_file_numbers = ["00", "01", "02", "03", "04", "05"]  # More files for more statements
    for i, request in enumerate(allowed_requests):
        if i < len(allowed_file_numbers):
            file_number = allowed_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved allowed request for {request.get('StatementSid', 'Unknown')} to {filepath}")
    
    # Save denied requests
    denied_file_numbers = ["93", "94", "95", "96", "97", "98"]  # More files for more statements
    for i, request in enumerate(denied_requests):
        if i < len(denied_file_numbers):
            file_number = denied_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved denied request for {request.get('StatementSid', 'Unknown')} to {filepath}")

def process_policy(policy_json, policy_index, output_dir):
    """Process a single policy and save requests to files - Original function"""
    print(f"Processing policy {policy_index}...")
    
    # Generate requests (3 allowed, 2 denied)
    requests = generate_labeled_requests(policy_json, num_allow=3, num_deny=2)
    
    # Separate allowed and denied requests
    allowed_requests = [req for req in requests if req['ExpectedDecision'] == 'Allow']
    denied_requests = [req for req in requests if req['ExpectedDecision'] == 'Deny']
    
    print(f"  Generated {len(allowed_requests)} allowed and {len(denied_requests)} denied requests")

    # Save allowed requests to files 90.json, 91.json, 92.json
    allowed_file_numbers = [90, 91, 92]
    for i, request in enumerate(allowed_requests):
        if i < len(allowed_file_numbers):
            file_number = allowed_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved allowed request to {filepath}")
    
    # Save denied requests to files 93.json, 94.json
    denied_file_numbers = [93, 94]
    for i, request in enumerate(denied_requests):
        if i < len(denied_file_numbers):
            file_number = denied_file_numbers[i]
            filepath = os.path.join(output_dir, f"{file_number}.json")
            save_request_to_file(request, filepath)
            print(f"  Saved denied request to {filepath}")

def main():
    parser = argparse.ArgumentParser(description='Generate requests for multiple policies')
    parser.add_argument('--policies', '-p', nargs='+', required=True, 
                       help='Policy JSON strings or file paths')
    parser.add_argument('--output-dir', '-o', 
                       default='/home/bhall2/Documents/fixmypolicy/FL/Dataset/requests',
                       help='Output directory for request files')
    parser.add_argument('--comprehensive', '-c', action='store_true',
                       help='Generate comprehensive requests covering all statements')
    
    args = parser.parse_args()
    
    # Create output directory if it doesn't exist
    os.makedirs(args.output_dir, exist_ok=True)
    
    for i, policy_input in enumerate(args.policies):
        # Check if it's a file path or JSON string
        if os.path.isfile(policy_input):
            with open(policy_input, 'r') as f:
                policy_json = f.read()
        else:
            policy_json = policy_input
        
        # Create subdirectory for this policy
        policy_dir = os.path.join(args.output_dir, f"policy_{i}")
        os.makedirs(policy_dir, exist_ok=True)
        
        try:
            if args.comprehensive:
                process_policy_comprehensive(policy_json, i, policy_dir)
            else:
                process_policy(policy_json, i, policy_dir)
        except Exception as e:
            print(f"Error processing policy {i}: {e}")
            continue
    
    print("All policies processed!")

# Example usage
if __name__ == "__main__":
    example_policy = '''
{
  "Statement": [
    {
      "Action": [
        "s3:Get*",
        "s3:List*",
        "s3:PutObject",
        "s3:DeleteObject"
      ],
      "Resource": "arn:aws:s3:::athena-query-results/*",
      "Effect": "Allow",
      "Sid": "AllowS3AccessToSaveAndReadQueryResults"
    },
    {
      "Action": [
        "s3:*"
      ],
      "Resource": "arn:aws:s3:::bkt_logs/*",
      "Effect": "Allow",
      "Sid": "AllowS3AccessForGlueToReadLogs"
    },
    {
      "Action": [
        "athena:GetQueryExecution",
        "athena:StartQueryExecution",
        "athena:StopQueryExecution",
        "athena:GetWorkGroup",
        "athena:GetDatabase",
        "athena:BatchGetQueryExecution",
        "athena:GetQueryResults",
        "athena:GetQueryResultsStream",
        "athena:GetTableMetadata"
      ], 
      "Resource": [
        "*"
      ],
      "Effect": "Allow",
      "Sid": "AllowAthenaAccess"
    },
    {
      "Action": [
        "glue:GetTable",
        "glue:GetDatabase",
        "glue:GetPartitions"
      ],
      "Resource": [
        "*"
      ],
      "Effect": "Allow",
      "Sid": "AllowGlueAccess"
    },
    {
      "Action": [
        "kms:CreateGrant",
        "kms:DescribeKey",
        "kms:Decrypt"
      ],
      "Resource": [
        "*"
      ],
      "Effect": "Allow",
      "Sid": "AllowKMSAccess"
    }
  ]
}
'''
    
    output_dir = '/home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2'
    os.makedirs(output_dir, exist_ok=True)
    
    # Use the comprehensive version to cover all statements
    process_policy_comprehensive(example_policy, 0, output_dir)

Processing policy 0 comprehensively...
  Generated 5 allowed and 2 denied requests
  Allowed requests cover statements: ['AllowS3AccessToSaveAndReadQueryResults', 'AllowS3AccessForGlueToReadLogs', 'AllowAthenaAccess', 'AllowGlueAccess', 'AllowKMSAccess']
  Denied requests cover statements: ['AllowS3AccessToSaveAndReadQueryResults', 'AllowS3AccessForGlueToReadLogs']
  Saved allowed request for AllowS3AccessToSaveAndReadQueryResults to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/00.json
  Saved allowed request for AllowS3AccessForGlueToReadLogs to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/01.json
  Saved allowed request for AllowAthenaAccess to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/02.json
  Saved allowed request for AllowGlueAccess to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/03.json
  Saved allowed request for AllowKMSAccess to /home/bhall2/Documents/fixmypolicy/FL/Dataset/requests/version2/04.jso

In [ ]:
class RequestGenerator:
    def __init__(self, policy: Dict[str, Any]):
        self.policy = policy
        
    def generate_must_allow_requests(self, num_requests):
        
        allowed_requests = []
        actions = self.policy.get("Statement", [{}])[0].get("Action", [])
        resources = self.policy.get("Statement", [{}])[0].get("Resource", [])
        
        for _ in range(num_requests):
            request = {
                "Effect": "Allow",
                "Action": random.choice(actions) if actions else "s3:GetObject",
                "Resource": random.choice(resources) if resources else "arn:aws:s3:::example-bucket/*",
                "Principal": "bob"
            }
            allowed_requests.append(request)
        return allowed_requests
    
    def generate_must_deny_requests(self, num_requests):
        #from the allowed requests, generate denied requests by modifying actions or resources
        denied_requests = []
        allowed_requests = self.generate_must_allow_requests(num_requests)
        for request in allowed_requests:
            denied_request = request.copy()
            # Change action to something not allowed
            denied_request["Action"] = "s3:DeleteObject"
            denied_requests.append(denied_request)
            denied_request["Resource"] = "arn:aws:s3:::example-bucket/forbidden/*"
            denied_requests.append(denied_request)
        return denied_requests

In [18]:
import json
import random
import uuid
from typing import Dict, List, Any, Optional

class RequestGenerator:
    def __init__(self, policy: Dict[str, Any]):
        self.policy = policy
        # More comprehensive action lists per service
        self.service_actions = {
            "s3": ["GetObject", "PutObject", "DeleteObject", "ListBucket", "GetBucketLocation", 
                  "PutBucketPolicy", "GetBucketAcl", "CreateBucket", "DeleteBucket"],
            "athena": ["GetQueryExecution", "StartQueryExecution", "StopQueryExecution", 
                      "GetWorkGroup", "GetDatabase", "BatchGetQueryExecution", "GetQueryResults",
                      "GetQueryResultsStream", "GetTableMetadata", "CreateWorkGroup", "DeleteWorkGroup"],
            "glue": ["GetTable", "GetDatabase", "GetPartitions", "CreateTable", "DeleteTable",
                    "UpdateTable", "CreateDatabase", "DeleteDatabase"],
            "kms": ["CreateGrant", "DescribeKey", "Decrypt", "Encrypt", "GenerateDataKey",
                   "DeleteAlias", "CreateKey", "ScheduleKeyDeletion"],
            "ec2": ["DescribeInstances", "RunInstances", "TerminateInstances", "CreateSecurityGroup"],
            "iam": ["CreateUser", "DeleteUser", "AttachUserPolicy", "ListUsers"],
            "lambda": ["InvokeFunction", "CreateFunction", "DeleteFunction", "UpdateFunctionCode"],
            "dynamodb": ["PutItem", "DeleteItem", "GetItem", "Scan", "Query", "CreateTable"]
        }
    
    def extract_policy_elements(self) -> Dict[str, List[str]]:
        """Extract actions, resources, and principals from the policy"""
        elements = {
            "actions": [],
            "resources": [],
            "principals": []
        }
        
        statements = self.policy.get("Statement", [])
        for statement in statements:
            if statement.get("Effect") == "Allow":
                # Extract actions
                actions = statement.get("Action", [])
                if isinstance(actions, str):
                    elements["actions"].append(actions)
                elif isinstance(actions, list):
                    elements["actions"].extend(actions)
                
                # Extract resources
                resources = statement.get("Resource", [])
                if isinstance(resources, str):
                    elements["resources"].append(resources)
                elif isinstance(resources, list):
                    elements["resources"].extend(resources)
                
                # Extract principals
                principal = statement.get("Principal")
                if isinstance(principal, str):
                    elements["principals"].append(principal)
                elif isinstance(principal, list):
                    elements["principals"].extend(principal)
        
        return elements
    
    def generate_denied_actions(self, allowed_actions: List[str]) -> List[str]:
        """Generate actions that should be denied"""
        denied_actions = set()
        
        # Get all services that have allowed actions
        allowed_services = set()
        for action in allowed_actions:
            if ":" in action:
                service = action.split(":", 1)[0]
                allowed_services.add(service)
        
        # For each service, find actions that aren't allowed
        for service in allowed_services:
            if service in self.service_actions:
                service_allowed = set()
                
                # Check what's actually allowed for this service
                for action in allowed_actions:
                    if action.startswith(f"{service}:"):
                        if action.endswith("*"):
                            # Wildcard - all actions for this service are allowed
                            service_allowed.update(self.service_actions[service])
                        else:
                            operation = action.split(":", 1)[1]
                            service_allowed.add(operation)
                
                # Find actions that aren't allowed
                for action in self.service_actions[service]:
                    if action not in service_allowed:
                        denied_actions.add(f"{service}:{action}")
        
        # Add actions from services not in the policy at all
        other_services = ["ec2", "iam", "lambda", "dynamodb", "rds", "sns", "sqs"]
        for service in other_services:
            if service not in allowed_services and service in self.service_actions:
                for action in self.service_actions[service][:2]:  # Just add a few
                    denied_actions.add(f"{service}:{action}")
        
        return list(denied_actions)
    
    def generate_denied_resources(self, allowed_resources: List[str]) -> List[str]:
        """Generate resources that should be denied"""
        denied_resources = set()
        
        for resource in allowed_resources:
            if resource == "*":
                # If wildcard, create specific resources that might be sensitive
                denied_resources.update([
                    "arn:aws:s3:::forbidden-bucket/*",
                    "arn:aws:iam::123456789012:role/admin-role",
                    "arn:aws:kms:us-east-1:123456789012:key/forbidden-key"
                ])
            elif "arn:aws:s3:::" in resource:
                # For S3 resources, create variations
                if resource.endswith("/*"):
                    bucket_name = resource.split(":::")[1].split("/")[0]
                    denied_resources.update([
                        f"arn:aws:s3:::different-{bucket_name}/*",
                        f"arn:aws:s3:::{bucket_name}-forbidden/*",
                        "arn:aws:s3:::completely-different-bucket/*"
                    ])
                else:
                    # Specific file
                    parts = resource.split("/")
                    if len(parts) > 1:
                        bucket_part = "/".join(parts[:-1])
                        file_part = parts[-1]
                        denied_resources.update([
                            f"{bucket_part}/forbidden-{file_part}",
                            f"{bucket_part.replace(':::', ':::forbidden-')}/{file_part}"
                        ])
            else:
                # Simple resource names
                denied_resources.update([
                    f"forbidden-{resource}",
                    f"{resource}-forbidden",
                    f"unauthorized/{resource}"
                ])
        
        return list(denied_resources)
    
    def expand_wildcard_action(self, action: str) -> str:
        """Convert wildcard actions to specific actions"""
        if not action.endswith("*"):
            return action
            
        service = action.split(":")[0]
        if service in self.service_actions:
            return f"{service}:{random.choice(self.service_actions[service])}"
        else:
            return action.replace("*", "GetObject")  # Default fallback
    
    def expand_wildcard_resource(self, resource: str) -> str:
        """Convert wildcard resources to specific resources"""
        if resource == "*":
            # Return a specific resource
            return random.choice([
                "arn:aws:s3:::my-bucket/document.txt",
                "arn:aws:athena:us-east-1:123456789012:workgroup/primary",
                "arn:aws:glue:us-east-1:123456789012:table/my-database/my-table"
            ])
        elif resource.endswith("/*"):
            base_path = resource[:-2]
            suffixes = ["/document.txt", "/data/file.json", "/logs/app.log", "/temp/upload.tmp"]
            return base_path + random.choice(suffixes)
        elif resource.endswith("*"):
            base_path = resource[:-1]
            suffixes = ["file1", "document", "data123"]
            return base_path + random.choice(suffixes)
        else:
            return resource
    
    def generate_multiple_actions(self, base_actions: List[str], count: int = None) -> List[str]:
        """Generate multiple related actions for a request"""
        if count is None:
            count = random.randint(1, 3)  # 1-3 actions per request
        
        actions = []
        available_actions = []
        
        # Expand all base actions to get a pool of specific actions
        for base_action in base_actions:
            if base_action.endswith("*"):
                service = base_action.split(":")[0]
                if service in self.service_actions:
                    for action in self.service_actions[service]:
                        available_actions.append(f"{service}:{action}")
            else:
                available_actions.append(base_action)
        
        # Remove duplicates and select random actions
        available_actions = list(set(available_actions))
        selected_count = min(count, len(available_actions))
        actions = random.sample(available_actions, selected_count)
        
        return actions
    
    def generate_multiple_resources(self, base_resources: List[str], count: int = None) -> List[str]:
        """Generate multiple related resources for a request"""
        if count is None:
            count = random.randint(1, 2)  # 1-2 resources per request
        
        resources = []
        
        for _ in range(count):
            if base_resources:
                base_resource = random.choice(base_resources)
                expanded_resource = self.expand_wildcard_resource(base_resource)
                resources.append(expanded_resource)
            else:
                resources.append("arn:aws:s3:::my-bucket/specific-file.txt")
        
        # Remove duplicates while preserving order
        seen = set()
        unique_resources = []
        for resource in resources:
            if resource not in seen:
                seen.add(resource)
                unique_resources.append(resource)
        
        return unique_resources

    def generate_must_allow_requests(self, num_requests: int) -> List[Dict[str, Any]]:
        """Generate requests that must be allowed by the policy"""
        allowed_requests = []
        policy_elements = self.extract_policy_elements()
        
        if not policy_elements["actions"]:
            raise ValueError("No allowed actions found in policy")
        
        for i in range(num_requests):
            # Generate multiple allowed actions
            actions = self.generate_multiple_actions(policy_elements["actions"])
            
            # Generate multiple allowed resources
            resources = self.generate_multiple_resources(policy_elements["resources"])
            
            # Select allowed principal
            principal = None
            if policy_elements["principals"]:
                principal = random.choice(policy_elements["principals"])
            
            request = {
                "id": f"allow_{uuid.uuid4().hex[:8]}",
                "Effect": "allow",
                "Action": actions,
                "Resource": resources
            }
            
            if principal:
                request["Principal"] = principal
            
            allowed_requests.append(request)
        
        return allowed_requests
    
    def generate_multiple_denied_actions(self, denied_actions: List[str], count: int = None) -> List[str]:
        """Generate multiple denied actions for a request"""
        if count is None:
            count = random.randint(1, 2)  # 1-2 denied actions per request
        
        if not denied_actions:
            return ["lambda:InvokeFunction"]  # Fallback
        
        selected_count = min(count, len(denied_actions))
        return random.sample(denied_actions, selected_count)
    
    def generate_multiple_denied_resources(self, denied_resources: List[str], count: int = None) -> List[str]:
        """Generate multiple denied resources for a request"""
        if count is None:
            count = random.randint(1, 2)  # 1-2 denied resources per request
        
        if not denied_resources:
            return ["arn:aws:s3:::forbidden-bucket/file.txt"]  # Fallback
        
        selected_count = min(count, len(denied_resources))
        return random.sample(denied_resources, selected_count)

    def generate_must_deny_requests(self, num_requests: int) -> List[Dict[str, Any]]:
        """Generate requests that must be denied by the policy"""
        denied_requests = []
        policy_elements = self.extract_policy_elements()
        
        # Generate denied variations
        denied_actions = self.generate_denied_actions(policy_elements["actions"])
        denied_resources = self.generate_denied_resources(policy_elements["resources"])
        
        for i in range(num_requests):
            # Strategy: alternate between denied action and denied resource
            if i % 2 == 0 and denied_actions:
                # Use denied actions with allowed resources
                actions = self.generate_multiple_denied_actions(denied_actions)
                resources = self.generate_multiple_resources(policy_elements["resources"])
                
            else:
                # Use allowed actions with denied resources
                if denied_resources:
                    actions = self.generate_multiple_actions(policy_elements["actions"])
                    resources = self.generate_multiple_denied_resources(denied_resources)
                else:
                    # Fallback to denied actions
                    actions = self.generate_multiple_denied_actions(denied_actions)
                    resources = ["arn:aws:s3:::forbidden-bucket/file.txt"]
            
            request = {
                "id": f"deny_{uuid.uuid4().hex[:8]}",
                "Effect": "deny",
                "Action": actions,
                "Resource": resources
            }
            
            denied_requests.append(request)
        
        return denied_requests
    
    def generate_all_requests(self, num_allow: int = 3, num_deny: int = 2) -> Dict[str, Any]:
        """Generate complete set of must-allow and must-deny requests"""
        try:
            must_allow = self.generate_must_allow_requests(num_allow)
            must_deny = self.generate_must_deny_requests(num_deny)
            
            # Combine all requests
            all_requests = must_allow + must_deny
            
            return {
                "Requests": all_requests
            }
        except Exception as e:
            return {
                "error": f"Failed to generate requests: {str(e)}"
            }

# Test with your policy
def main():
    policy = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal":"bob",
      "Action": ["S3:GetObject", "S3:ListBucket", "S3:PutObject"],
      "Resource": "foobar/*",
      "Condition": {
        "NumericLessThanEquals": {
          "s3:MaxKeys": 100
        }
      }
    }
  ]
}

    generator = RequestGenerator(policy)
    test_data = generator.generate_all_requests(num_allow=3, num_deny=2)
    
    print(json.dumps(test_data, indent=2))
    return test_data

if __name__ == "__main__":
    main()

{
  "Requests": [
    {
      "id": "allow_25f13aaa",
      "Effect": "allow",
      "Action": [
        "S3:PutObject",
        "S3:ListBucket"
      ],
      "Resource": [
        "foobar/logs/app.log",
        "foobar/temp/upload.tmp"
      ],
      "Principal": "bob"
    },
    {
      "id": "allow_64d27fce",
      "Effect": "allow",
      "Action": [
        "S3:PutObject",
        "S3:GetObject",
        "S3:ListBucket"
      ],
      "Resource": [
        "foobar/document.txt"
      ],
      "Principal": "bob"
    },
    {
      "id": "allow_deded197",
      "Effect": "allow",
      "Action": [
        "S3:ListBucket",
        "S3:GetObject"
      ],
      "Resource": [
        "foobar/temp/upload.tmp",
        "foobar/document.txt"
      ],
      "Principal": "bob"
    },
    {
      "id": "deny_e9401c26",
      "Effect": "deny",
      "Action": [
        "ec2:RunInstances"
      ],
      "Resource": [
        "foobar/temp/upload.tmp",
        "foobar/logs/app.log"
      ]
    

In [1]:
import json

principals = ["bob", "alice"]
actions    = ["S3:GetObject", "S3:PutObject"]
resources  = ["foobar", "foobar/object.txt"]
conditions = [
    {"s3:MaxKeys": 50},   # < 100
    {"s3:MaxKeys":100},   # == 100
    {"s3:MaxKeys":101},   # > 100
    None                  # missing
]

requests = []
for p in principals:
    for a in actions:
        for r in resources:
            for c in conditions:
                req = {
                    "Principal": p,
                    "Action":    a,
                    "Resource":  r
                }
                if c is not None:
                    req["Condition"] = {"NumericLessThanEquals": c}
                requests.append(req)

print(json.dumps(requests, indent=2))


[
  {
    "Principal": "bob",
    "Action": "S3:GetObject",
    "Resource": "foobar",
    "Condition": {
      "NumericLessThanEquals": {
        "s3:MaxKeys": 50
      }
    }
  },
  {
    "Principal": "bob",
    "Action": "S3:GetObject",
    "Resource": "foobar",
    "Condition": {
      "NumericLessThanEquals": {
        "s3:MaxKeys": 100
      }
    }
  },
  {
    "Principal": "bob",
    "Action": "S3:GetObject",
    "Resource": "foobar",
    "Condition": {
      "NumericLessThanEquals": {
        "s3:MaxKeys": 101
      }
    }
  },
  {
    "Principal": "bob",
    "Action": "S3:GetObject",
    "Resource": "foobar"
  },
  {
    "Principal": "bob",
    "Action": "S3:GetObject",
    "Resource": "foobar/object.txt",
    "Condition": {
      "NumericLessThanEquals": {
        "s3:MaxKeys": 50
      }
    }
  },
  {
    "Principal": "bob",
    "Action": "S3:GetObject",
    "Resource": "foobar/object.txt",
    "Condition": {
      "NumericLessThanEquals": {
        "s3:MaxKeys": 100
    